# 🐾 Animal Diet Classifier — Entrenamiento en Google Colab Pro+

Entrena los dos modelos de la cascada (**ResNet-18** primario y **ResNet-50** respaldo) aprovechando la GPU de Colab Pro+ (idealmente **A100**), con **precisión mixta (AMP)** y carga de datos en paralelo.

**Antes de empezar:**
1. Menú `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU` (elige **A100** si está disponible; si no, L4 o V100).
2. Sube el dataset comprimido a tu Google Drive (ver paso 4).

**Flujo:** verificar GPU → montar Drive → clonar repo → instalar → descomprimir dataset → entrenar ResNet-18 → entrenar ResNet-50 → guardar pesos en Drive.

## 1. Verificar la GPU asignada

In [ ]:
!nvidia-smi
import torch
print('\nPyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memoria total: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Montar Google Drive

Aquí vivirá el dataset comprimido (entrada) y se guardarán los pesos entrenados (salida).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Carpeta de trabajo en tu Drive (ajústala si quieres otra ruta)
DRIVE_DIR = '/content/drive/MyDrive/animal_diet_classifier'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'weights'), exist_ok=True)
print('Carpeta en Drive:', DRIVE_DIR)

## 3. Clonar el repositorio

Si el repo es **privado**, reemplaza la URL por una con token:
`https://<TOKEN>@github.com/ErickJester/animal_diet_classifier.git`
(crea el token en GitHub → Settings → Developer settings → Personal access tokens).

In [ ]:
%cd /content
REPO_URL = 'https://github.com/ErickJester/animal_diet_classifier.git'
REPO_DIR = '/content/animal_diet_classifier'

import os
if os.path.isdir(REPO_DIR):
    print('El repo ya existe, actualizando...')
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL
    %cd $REPO_DIR

!git log --oneline -3

## 4. Instalar dependencias

Colab ya trae `torch`, `torchvision` y `numpy` con CUDA — **no los reinstalamos** para no romper la build de GPU. Solo añadimos lo que falte (OpenCV, para la prueba de inferencia).

In [ ]:
!pip install -q opencv-python-headless
print('Listo.')

## 5. Subir y descomprimir el dataset

El dataset (~31k imágenes) es demasiado grande para subirlo a mano cada vez. El camino recomendado:

**En tu PC (una sola vez)** — comprime la carpeta `dataset/` y súbela a Drive en `MyDrive/animal_diet_classifier/`:

```powershell
# PowerShell, dentro de D:\animal_diet_classifier
Compress-Archive -Path dataset -DestinationPath dataset.zip
```

Luego sube `dataset.zip` a `MyDrive/animal_diet_classifier/` (web de Drive o app de escritorio).

La siguiente celda copia el zip de Drive al disco **local** de Colab (SSD, mucho más rápido que leer de Drive durante el entreno) y lo descomprime dentro del repo.

In [ ]:
import os, shutil, time

ZIP_IN_DRIVE = os.path.join(DRIVE_DIR, 'dataset.zip')
assert os.path.isfile(ZIP_IN_DRIVE), f'No encuentro {ZIP_IN_DRIVE}. Sube dataset.zip a tu Drive.'

# 1) copiar zip a disco local de Colab
t = time.time()
shutil.copy(ZIP_IN_DRIVE, '/content/dataset.zip')
print('Copiado a local en %.0fs' % (time.time() - t))

# 2) descomprimir dentro del repo (queda /content/animal_diet_classifier/dataset/...)
%cd $REPO_DIR
!rm -rf dataset && unzip -q /content/dataset.zip -d /content/animal_diet_classifier/
print('Descomprimido.')

In [ ]:
# Verificar el conteo por clase (sanity check antes de entrenar)
from pathlib import Path
exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
base = Path('/content/animal_diet_classifier/dataset')
total = 0
for split in ('train', 'val'):
    print(f'\n[{split}]')
    for diet in ('carnivore', 'herbivore', 'omnivore'):
        d = base / split / diet
        n = sum(1 for f in d.rglob('*') if f.suffix.lower() in exts) if d.is_dir() else 0
        total += n
        print(f'  {diet:<11} {n:>7}')
print(f'\nTOTAL: {total} imágenes')
assert total > 0, 'No se encontraron imágenes. Revisa el zip.'

## 6. Entrenar el modelo primario (ResNet-18)

Rápido y liviano. Con AMP + workers + `channels_last`/TF32, en A100 cada época vuela.

**Entrenamiento en 2 fases** (`--two-phase`):
1. **Fase 1** — congela el backbone y entrena solo la cabeza `fc` (LR alto, `ReduceLROnPlateau`). Estabiliza la cabeza antes de tocar pesos pre-entrenados.
2. **Fase 2** — descongela los últimos `--unfreeze-n` bloques residuales y hace fine-tuning con LR bajo (`CosineAnnealingLR`).

El **mejor modelo se elige por Macro F1** de validación (no por accuracy), más honesto con clases desbalanceadas. `--epochs` se reparte: `--phase1-epochs` van a la Fase 1, el resto a la Fase 2.

**Ajusta `--batch-size` según la GPU:** A100 40 GB → 256–512 · L4 24 GB → 256 · V100 16 GB → 128. Si ves *out of memory*, bájalo.

In [ ]:
%cd $REPO_DIR
# Entrenamiento en 2 fases: primero la cabeza (backbone congelado), luego
# fine-tuning de los últimos bloques. El mejor modelo se elige por Macro F1.
!python train_classifier.py \
    --arch resnet18 \
    --epochs 30 \
    --two-phase \
    --phase1-epochs 10 \
    --phase1-lr 1e-3 \
    --phase2-lr 1e-5 \
    --unfreeze-n 2 \
    --batch-size 256 \
    --num-workers 4 \
    --amp \
    --device cuda \
    --output weights/diet_resnet18.pt

## 7. Entrenar el modelo de respaldo (ResNet-50)

Más preciso y pesado. Usa un batch algo menor (es ~4× el cómputo de ResNet-18).

In [ ]:
%cd $REPO_DIR
# ResNet-50: mismo esquema de 2 fases con batch menor (es ~4x el cómputo).
!python train_classifier.py \
    --arch resnet50 \
    --epochs 30 \
    --two-phase \
    --phase1-epochs 10 \
    --phase1-lr 1e-3 \
    --phase2-lr 1e-5 \
    --unfreeze-n 2 \
    --batch-size 128 \
    --num-workers 4 \
    --amp \
    --device cuda \
    --output weights/diet_resnet50.pt

## 8. Guardar los pesos en Drive

El disco de Colab es efímero: copia los `.pt` a tu Drive para no perderlos al cerrar la sesión.

In [ ]:
import shutil, os, glob
dst = os.path.join(DRIVE_DIR, 'weights')
os.makedirs(dst, exist_ok=True)
for pt in glob.glob('/content/animal_diet_classifier/weights/*.pt'):
    shutil.copy(pt, dst)
    print('Guardado:', os.path.join(dst, os.path.basename(pt)))
print('\nPesos en Drive:', dst)

## 9. (Opcional) Prueba rápida de inferencia

Clasifica una imagen de validación con la cascada completa para verificar que los pesos cargan bien.

In [ ]:
%cd $REPO_DIR
from pathlib import Path
from classifier import DietClassifier

clf = DietClassifier(device='cuda')
print('Clasificador disponible:', clf.is_available)

# toma una imagen de ejemplo de cada clase en val/
base = Path('dataset/val')
for diet in ('carnivore', 'herbivore', 'omnivore'):
    sample = next((f for f in (base / diet).glob('*') if f.suffix.lower() in {'.jpg','.jpeg','.png'}), None)
    if sample:
        r = clf.classify_image(str(sample))
        print(f'{diet:<11} (real)  →  {r.label:<11} {r.confidence*100:5.1f}%  [{r.source}]')

---
### Notas
- **2 fases** (`--two-phase`): Fase 1 entrena solo la cabeza con el backbone congelado (LR alto, `ReduceLROnPlateau`); Fase 2 descongela los últimos `--unfreeze-n` bloques y afina con LR bajo (`CosineAnnealingLR`). Si prefieres el entrenamiento clásico de una sola fase, omite `--two-phase` y usa `--lr`.
- **Mejor modelo por Macro F1**: el checkpoint se guarda cuando mejora el F1 macro de validación, no la accuracy. Al final se imprimen accuracy global, Macro F1 y la matriz de confusión por clase.
- **AMP** (`--amp`) usa precisión mixta FP16/FP32: hasta ~2× más rápido y menos memoria en GPUs con Tensor Cores (A100/V100/L4), sin pérdida apreciable de exactitud.
- **`channels_last` + TF32 + `cudnn.benchmark`** se activan solos en CUDA: 5–15% extra de velocidad sin tocar nada.
- **`--num-workers 4`** carga las imágenes en paralelo mientras la GPU entrena; evita que la GPU espere al disco. En Colab verás barras `tqdm` por época si está instalado.
- Si el entreno se corta por límite de tiempo de Colab, el **mejor** checkpoint (por F1) ya está en `weights/`. Vuelve a montar Drive y re-copia.
- Para iterar rápido en pruebas, baja `--epochs` (p.ej. 6 con `--phase1-epochs 3`) y revisa la matriz de confusión final.